# Chapter 2: Attention Mechanisms and Positional Encoding
**Module 04 – Introduction to LLMs in Python**

> *Instructor: Iván Palomares Carrascosa, PhD — Senior Data Science & AI Manager*

## 2.1 Why Attention Mechanisms?

RNNs process text **sequentially** — they struggle to retain information from far back in a long sequence.

**Attention** solves this by allowing the model to focus on **any part of the sequence** regardless of distance:

> Example: In "The animal didn't cross the street because **it** was too tired",
> attention helps the model connect "it" → "animal" even though they're far apart.

### Self-Attention
Each token **attends to all other tokens** in the same sequence, computing weighted relationships.

## 2.2 Positional Encoding

Transformers process all tokens **in parallel** (not sequentially), so they have no inherent sense of word order.

**Positional encoding** adds position information to each token embedding:
- Create a positional encoding vector **PE** for each position
- Based on **sine and cosine** functions at different frequencies
- Add PE to the token embedding E: `input = E + PE`

In [ ]:
import torch
import torch.nn as nn
import math
import matplotlib.pyplot as plt

class PositionalEncoder(nn.Module):
    def __init__(self, d_model, max_seq_length=512):
        """
        Args:
            d_model: embedding/model dimension
            max_seq_length: maximum sequence length supported
        """
        super(PositionalEncoder, self).__init__()
        self.d_model = d_model

        # Create positional encoding matrix: (max_seq_length, d_model)
        pe = torch.zeros(max_seq_length, d_model)

        # Position indices: (max_seq_length, 1)
        position = torch.arange(0, max_seq_length, dtype=torch.float).unsqueeze(1)

        # Scaling term
        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * -(math.log(10000.0) / d_model)
        )

        # Apply sine to even dimensions, cosine to odd
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        # Add batch dimension: (1, max_seq_length, d_model)
        pe = pe.unsqueeze(0)

        # Register as non-trainable buffer (saved with model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        """Add positional encoding to input embeddings."""
        x = x + self.pe[:, :x.size(1)]
        return x


# Test the positional encoder
pe = PositionalEncoder(d_model=64, max_seq_length=100)
print(f"PE buffer shape: {pe.pe.shape}")  # (1, 100, 64)

# Visualize positional encodings
plt.figure(figsize=(10, 4))
plt.imshow(pe.pe.squeeze(0).detach().numpy(), aspect='auto', cmap='RdBu')
plt.colorbar()
plt.title('Positional Encoding Matrix')
plt.xlabel('Embedding Dimension')
plt.ylabel('Position in Sequence')
plt.show()

## 2.3 Self-Attention Mechanism

Self-attention computes three matrices from the input:
- **Q (Query)** — what am I looking for?
- **K (Key)** — what do I contain?
- **V (Value)** — what do I represent?

$$\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

In [ ]:
class SelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(SelfAttention, self).__init__()
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=num_heads,
            batch_first=True
        )

    def forward(self, x):
        # Q = K = V = x  (self-attention)
        attn_output, attn_weights = self.attention(x, x, x)
        return attn_output, attn_weights


# Test
d_model = 64
seq_len = 10
batch_size = 2

self_attn = SelfAttention(d_model=d_model, num_heads=8)
x = torch.randn(batch_size, seq_len, d_model)
out, weights = self_attn(x)

print(f"Input shape:   {x.shape}")
print(f"Output shape:  {out.shape}")
print(f"Weights shape: {weights.shape}")  # attention map

## Summary

| Component | Purpose |
|---|---|
| Positional Encoding | Injects word-order information into parallel processing |
| Self-Attention | Lets each token attend to all others |
| Q, K, V | Learned projections for computing attention scores |
| Multi-Head Attention | Multiple attention heads learn different relationships |